<!-- ## New Approach

So I've been mulling over this task, and I think it'd be great if we could take our conditional entropy algorithmic approach and ML approaches and combine them. So I Googled for a bit, and this is what's broadly known as differentiable programming. Differentiable programming is simply the idea that you can take an algorithm that was traditionally non-differentiable and reformulate it in a continuous way so that you can optimize the whole process using gradient descent. I found a paper that does exactly this for structured prediction tasks (https://arxiv.org/pdf/1802.03676), which our approach will be based on.

The plan here is to use neural scoring functions to assign a quality score to every possible candidate segment in a word. These functions are just lightweight neural networks (like an MLP) that take in features from our main model. Our approach is as follows:

We first run a character-level LSTM over the input word to generate hidden representations at every position. Then, for any candidate segment spanning positions $i$ to $j$ (within a set maximum length), we compute a score $s(i,j)$ by feeding the concatenation of the hidden state at $i$ and the hidden state at $j-1$ into our neural scoring function. These scores capture how plausible it is that the segment from $i$ to $j$ forms a valid morpheme (or at least, we'll train the model to make them do so).

Next, instead of deciding on segmentation boundaries independently, we plug all these segment scores into a differentiable dynamic programming algorithm that efficiently computes the global score of the segmentation and the corresponding partition function $Z$ (see: https://arxiv.org/pdf/1802.03676). Here "global score" refers to the sum of the scores of all segments in the segmentation. The partition function $Z$ is the sum of the scores of all possible segmentations of the word. Our loss is then defined as

$$
\mathcal{L} = \log Z - s_{\text{gold}},
$$

where $s_{\text{gold}}$ is the score assigned to the correct (gold) segmentation. By minimizing this loss, we encourage the model to assign higher scores to correct segmentations relative to all possible segmentations. Crucially, because the entire process - from the LSTM encoding to the neural scoring and through the dynamic programming module - is differentiable, we can train the whole system end-to-end with our favorite gradient-based optimizer. -->
## New Approach

So I've been mulling over this task, and I think it'd be great if we could take our conditional entropy algorithmic approach and ML approaches and combine them. So I Googled for a bit, and this is what's broadly known as differentiable programming. Differentiable programming is simply the idea that you can take an algorithm that was traditionally non-differentiable and reformulate it in a continuous way so that the entire process can be optimized using gradient descent. For our segmentation problem, I found a paper that does exactly this for structured prediction tasks (https://arxiv.org/pdf/1802.03676), which our approach will be based on.

The plan is to use a neural scoring function to assign a quality score to every possible candidate segment in a word. Concretely, we perform the following steps:

1. **Encoding:**  
   We run a character-level LSTM over the input word, obtaining hidden representations $h_1, h_2, \dots, h_n$ at each position.

2. **Scoring Segments:**  
   For any candidate segment spanning positions $i$ to $j$ (with $j - i \leq \textrm{len}(\text{word})$), we compute a score  
   $$
   s(i,j) = \text{MLP}(h_i \oplus h_{j-1}),
   $$
   where $\oplus$ denotes vector concatenation. This score reflects how plausible it is that the substring $\text{word}[i:j]$ forms a valid morpheme (or at least, we'll train the model to make it do so).

3. **Global Segmentation Score:**  
   We'll use a differentiable dynamic programming algorithm to compute the global score of the segmentation and the corresponding partition function $Z$. Here, “global” means that the score aggregates the candidate segment scores along a complete segmentation path, ensuring that the decisions in one part of the word are optimally consistent with those in the rest. Let $Z$ denote the partition function:
   $$
   Z = \sum_{S} \exp(s(S)),
   $$
   so that
   $$
   \log Z = \log\!\left(\sum_{S} \exp(s(S))\right).
   $$
   This $Z$ aggregates the scores of all possible segmentations, thereby considering the overall quality of a complete segmentation path rather than individual segment scores in isolation.

4. **Loss Function:**  
   Given the score $s_{\text{gold}}$ for the correct (gold) segmentation, we define the loss as
   $$
   \mathcal{L} = \log Z - s_{\text{gold}}.
   $$
   Minimizing this loss encourages the model to assign a higher global score to the correct segmentation relative to all possible segmentations.

Because every step from the LSTM encoding to the DP aggregation is differentiable, we can train the entire system end-to-end via backpropagation.

In [5]:
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from eval import run_eval
from preprocess import clean_target_segmentation

In [6]:
def load_lines(filepath):
    with open(filepath, encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

# load data
splits = ['train', 'dev', 'test']       
data = {}

# Set this flag to enable/disable "cleaning" of target segmentations
# e.g. remove additional characters from target segmentations

PREPROCESS_TGT = True

for lang in ['shp', 'tar']:
    data[lang] = {}
    for split in splits:
        data[lang][split] = {}
        
        # load source file (unsegmented words)
        src_file = os.path.join('dataset', f'{lang}.{split}.src')
        if os.path.exists(src_file):
            data[lang][split]['src'] = load_lines(src_file)
            
        # load target file (morphological segmentations) if exists
        tgt_file = os.path.join('dataset', f'{lang}.{split}.tgt')
        if os.path.exists(tgt_file):
            data[lang][split]['tgt'] = load_lines(tgt_file)

if PREPROCESS_TGT:
    for lang in data:
        for split in data[lang]:
            if 'src' in data[lang][split] and 'tgt' in data[lang][split]:
                new_tgt = []
                for src_word, tgt_line in zip(data[lang][split]['src'], data[lang][split]['tgt']):
                    cleaned = clean_target_segmentation(src_word, tgt_line)
                    new_tgt.append(" ".join(cleaned))
                data[lang][split]['tgt'] = new_tgt
    print("Target segmentations cleaned.")

Target segmentations cleaned.


In [3]:
# check data is loaded correctly

for lang in ['shp', 'tar']:
    for split in splits:
        if 'src' in data[lang][split] and 'tgt' in data[lang][split]:
            assert len(data[lang][split]['src']) == len(data[lang][split]['tgt'])
    print(f'{lang} is loaded correctly.')
        
print()
# print example for both languages of src and tgt

for lang in ['shp', 'tar']:
    for split in ['train']:
        print(f'{lang}.{split}')
        for i in range(3):
            print(f'    {data[lang][split]["src"][i]}')
            print(f'    {data[lang][split]["tgt"][i]}')
            print()

shp is loaded correctly.
tar is loaded correctly.

shp.train
    yoyoaibata
    yoyoa ibat a

    kotsatax
    kotsat ax

    bokasai
    bo kas ai

tar.train
    páa
    páa

    pochítisi
    pochí ti si

    konári
    ko nári



In [7]:
class MorphDataset(Dataset):
    def __init__(self, src, tgt, char2idx, max_len):
        self.src = src
        self.tgt = tgt
        self.char2idx = char2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src_word = self.src[idx]
        tgt_segmentation = self.tgt[idx]

        src_indices = [self.char2idx.get(char, self.char2idx['<UNK>']) for char in src_word]
        
        labels = [0] * len(src_word)
        tgt_list = tgt_segmentation.split()
        char_idx = 0
        for morpheme in tgt_list:
            labels[char_idx] = 1
            char_idx += len(morpheme)
            
        src_indices = src_indices[:self.max_len]
        labels = labels[:self.max_len]

        src_indices += [self.char2idx['<PAD>']] * (self.max_len - len(src_indices))
        labels += [0] * (self.max_len - len(labels))

        return {
            'src_indices': torch.tensor(src_indices, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long)
        }
        
def create_char2idx(data, lang):
    char2idx = {'<PAD>': 0, '<UNK>': 1}
    for split in splits:
        if 'src' in data[lang][split]:
            for word in data[lang][split]['src']:
                for char in word:
                    if char not in char2idx:
                        char2idx[char] = len(char2idx)
    return char2idx

def create_idx2char(char2idx):
    return {idx: char for char, idx in char2idx.items()}

In [8]:
# Set maximum sequence length
max_len = max(len(word) for lang in ['shp', 'tar'] for split in splits if 'tgt' in data[lang][split] for word in data[lang][split]['tgt'])
batch_size = 32

train_datasets = {}
dev_datasets = {}
test_datasets = {}
char2idx_dict = {}
train_loaders = {}
dev_loaders = {}
test_loaders = {}

for lang in ['shp', 'tar']:
    char2idx = create_char2idx(data, lang)
    char2idx_dict[lang] = char2idx

    train_datasets[lang] = MorphDataset(data[lang]['train']['src'], data[lang]['train']['tgt'], char2idx, max_len)
    train_loaders[lang] = DataLoader(train_datasets[lang], batch_size=batch_size, shuffle=True)
    
    dev_datasets[lang] = MorphDataset(data[lang]['dev']['src'], data[lang]['dev']['tgt'], char2idx, max_len)
    dev_loaders[lang] = DataLoader(dev_datasets[lang], batch_size=batch_size, shuffle=False)

    dummy_tgt = [""] * len(data[lang]['test']['src'])
    test_datasets[lang] = MorphDataset(data[lang]['test']['src'], dummy_tgt, char2idx, max_len)
    test_loaders[lang] = DataLoader(test_datasets[lang], batch_size=batch_size, shuffle=False)

    print(f"Data loaders created for {lang}.")
    print(f"Vocabulary size for {lang}: {len(char2idx)}")

Data loaders created for shp.
Vocabulary size for shp: 50
Data loaders created for tar.
Vocabulary size for tar: 35


In [9]:
class Segmenter(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, max_seg_len, char2idx, idx2char):
        super().__init__()
        self.pad_idx = char2idx['<PAD>']
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=self.pad_idx)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, bidirectional=True, batch_first=True)
        # scoring MLP
        self.mlp = nn.Sequential(nn.Linear(4*hidden_dim, 128),
                                 nn.ReLU(), 
                                 nn.Linear(128, 1))
        self.idx2char = idx2char

    def segment_score(self, h, i, j):
        seg_rep = torch.cat([h[i], h[j-1]], dim=-1)
        return self.mlp(seg_rep).squeeze()
    
    def viterbi_decode(self, h):
        L = h.size(0)
        dp = [None]*(L+1)           # dp[i] stores the best score for the first i characters
        bp = [None]*(L+1)           # bp[i] stores the best boundary before the i-th character
        dp[0] = 0.0
        
        # dp[i] = max(dp[j] + score(j, i)) for 0 <= j < i
        
        for i in range(1, L+1):
            best_score = -float('inf')
            best_j = None

            for j in range(0, i):
                score = dp[j] + self.segment_score(h, j, i).item()
                if score > best_score:
                    best_score = score
                    best_j = j
            dp[i] = best_score
            bp[i] = best_j
        
        boundaries = []
        i = L
        while i > 0:
            boundaries.append(bp[i])
            i = bp[i]
        
        boundaries = boundaries[::-1]
        boundaries.append(L)
        
        return boundaries
    
    def get_gold_boundaries(self, gold_labels):
        boundaries = [0]
        l = gold_labels.size(0)
        
        for i in range(1, l):
            if gold_labels[i].item() == 1:
                boundaries.append(i)
        
        if boundaries[-1] != l:
            boundaries.append(l)
        
        return boundaries

    def decode_word(self, h, src_word):
        boundaries = self.viterbi_decode(h)
        chars = [self.idx2char[idx.item()] for idx in src_word if idx.item() != self.pad_idx]
        segments = []
        
        for s, e in zip(boundaries, boundaries[1:]):
            segments.append("".join(chars[s:e]))
        
        return " ".join(segments)
    
    def dp_loss(self, h, gold_boundaries):
        L = h.size(0)
        dp = [None]*(L+1)
        dp[0] = torch.tensor(0.0, device=h.device)
        
        for i in range(1, L+1):
            candidates = []
            for j in range(0, i):
                score = self.segment_score(h, j, i)
                candidates.append(dp[j] + score)
                
            dp[i] = torch.logsumexp(torch.stack(candidates), dim=0)
        
        logZ = dp[L]
        gold_score = 0.0
        gold_boundaries_full = gold_boundaries[:]
        
        if gold_boundaries_full[-1] != L:
            gold_boundaries_full.append(L)
        
        for idx in range(1, len(gold_boundaries_full)):
            j = gold_boundaries_full[idx-1]
            i = gold_boundaries_full[idx]
            gold_score = gold_score + self.segment_score(h, j, i)
        
        return logZ - gold_score

    def forward(self, src, labels=None):
        emb = self.emb(src)
        out, _ = self.lstm(emb)
        
        if labels is not None:
            loss = 0.0    
            for b in range(src.size(0)):
                l = (src[b] != self.pad_idx).sum().item()
                h = out[b, :l, :]
                gold_b = self.get_gold_boundaries(labels[b, :l])
                loss = loss + self.dp_loss(h, gold_b)
            return loss / src.size(0)
        else:
            preds = []
            for b in range(src.size(0)):
                l = (src[b] != self.pad_idx).sum().item()
                h = out[b, :l, :]
                preds.append(self.decode_word(h, src[b]))
            return preds


In [11]:
def evaluate(model, loader):
    model.eval()
    golds = []
    preds = []
    
    with torch.no_grad():
    
        for batch in loader:
            src_indices = batch['src_indices'].to(device)
            labels = batch['labels'].to(device)
            batch_preds = model(src_indices)
    
            for b in range(src_indices.size(0)):
                l = (src_indices[b] != model.pad_idx).sum().item()
                
                gold_boundaries = []
                word = [model.idx2char[idx.item()] for idx in src_indices[b][:l]]
    
                for i in range(l):
                    if i == 0 or labels[b][i].item() == 1:
                        gold_boundaries.append(i)
    
                if gold_boundaries[-1] != l:
                    gold_boundaries.append(l)
    
                gold_seg = []
    
                for s, e in zip(gold_boundaries, gold_boundaries[1:]):
                    gold_seg.append("".join(word[s:e]))
    
                golds.append(" ".join(gold_seg))
                preds.append(batch_preds[b])
    
    model.train()
    
    f1 = run_eval(golds, preds)
    
    return f1

In [12]:
def train_model(model, train_loader, dev_loader, n_epochs=10, lr=0.001):
    opt = optim.Adam(model.parameters(), lr=lr)
    model.train()
    
    pbar = tqdm(total=n_epochs, desc="Epochs", position=0)
    for epoch in range(1, n_epochs+1):
        total_loss = 0.0
        
        for batch in train_loader:
            src_indices = batch['src_indices'].to(device)
            labels = batch['labels'].to(device)
            
            opt.zero_grad()
            loss = model(src_indices, labels)
            
            loss.backward()
            opt.step()
            total_loss += loss.item()
    
        avg_loss = total_loss / len(train_loader)
    
        # print(f'epoch {epoch} loss: {avg_loss:.4f}')
        pbar.desc = f"Epoch {epoch}/{n_epochs}, Avg Loss: {avg_loss:.4f}"
        pbar.update(1)
        
        evaluate(model, dev_loader)
        
    return model

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

emb_dim = 64
hidden_dim = 128
max_seg_len = 10

models = {}
for lang in ['shp', 'tar']:
    vocab_size = len(char2idx_dict[lang])
    idx2char = create_idx2char(char2idx_dict[lang])
    models[lang] = Segmenter(vocab_size, emb_dim, hidden_dim, max_seg_len, char2idx_dict[lang], idx2char).to(device)
    print(f'model created for {lang}, vocab size: {vocab_size}')

n_epochs = 25
for lang in ['shp', 'tar']:
    print(f'\ntraining model for {lang}')
    models[lang] = train_model(models[lang], train_loaders[lang], dev_loaders[lang], n_epochs=n_epochs)
    
dev_results = {}
for lang in ['shp', 'tar']:
    dev_results[lang] = evaluate(models[lang], dev_loaders[lang])
    print(f'{lang} dev F1 score: {dev_results[lang]:.4f}')
    
# v1
# shp dev F1 score: 0.7426
# tar dev F1 score: 0.7302

# v2
# shp dev F1 score: 0.7763
# tar dev F1 score: 0.7228

model created for shp, vocab size: 50
model created for tar, vocab size: 35

training model for shp


Epoch 25/25, Avg Loss: 0.0226: 100%|██████████| 25/25 [02:51<00:00,  6.87s/it]



training model for tar


Epoch 25/25, Avg Loss: 0.0044: 100%|██████████| 25/25 [02:09<00:00,  5.16s/it]


shp dev F1 score: 0.7854
tar dev F1 score: 0.7493


In [ ]:
def predict_and_save(model, loader, out_path):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in loader:
            src_indices = batch['src_indices'].to(device)
            batch_preds = model(src_indices)
            predictions.extend(batch_preds)

    with open(out_path, 'w', encoding='utf-8') as f:
        for line in predictions:
            f.write(line + "\n")
            
output_dir = 'segmenter_predictions'
os.makedirs(output_dir, exist_ok=True)

for lang in ['shp', 'tar']:
    out_path = os.path.join(output_dir, f'pred_{lang}.test.tgt')
    predict_and_save(models[lang], test_loaders[lang], out_path)
    print(f'predictions saved for {lang}')

predictions saved for shp
predictions saved for tar
